<a id="top"></a>

# Demo: build one node -- your first LangGraph graph

```yaml
title: "Demo: build one node -- your first LangGraph graph"
keywords: langgraph, StateGraph, node, typed state, typeddict, state update, pure function, compile, invoke, stream, extract, structured output, defensive parsing, ChatAnthropic, teacher demo
estimated duration: 30
```

> **Lesson:** L03. The runnable spine of the lesson -- covers Demos 1 & 2 in the
> [roadmap](../../../../docs/origin/lesson_roadmaps/L03/demos_or_activities.md).
> Makes **live** Claude calls -- set `ANTHROPIC_API_KEY` to run the node (the graph *build*,
> the *diagram*, and the offline before/after cells all run without a key). **Anchor model: Claude Sonnet 4.6 -- and only Sonnet.**
> Clear outputs before committing.

## Contents

- [1. Setup -- client and sample tickets](#1-setup----client-and-sample-tickets)
- [2. The "before": a plain L01/L02-style call](#2-the-before-a-plain-l01l02-style-call)
- [3. The typed state](#3-the-typed-state)
- [4. The node: one LLM call, wrapped](#4-the-node-one-llm-call-wrapped)
- [5. Compile the smallest possible graph](#5-compile-the-smallest-possible-graph)
- [6. Invoke and inspect one node](#6-invoke-and-inspect-one-node)
- [7. Watch it run with `stream()`](#7-watch-it-run-with-stream)
- [8. Takeaways](#8-takeaways)

## 1. Setup -- client and sample tickets

Two things, both self-contained: the **native LangChain `ChatAnthropic`** client (one of it --
a single Sonnet, used by the single node), and a few sample **support tickets** to extract
fields from.

**Notice the departure:** this is your *first framework lesson*. From here on, nodes call
`ChatAnthropic` directly, **not** the `potato_llm` seam from L01-L02 -- *frameworks bring their
own client abstraction.* The API key still loads through `common/config.py`; only the client
changed. The client is built only when a key is present, so a keyless restart-and-run-all still
passes -- the run cells below are gated on `LIVE`.

In [ ]:
import json
import re
from typing import TypedDict

from langchain_anthropic import ChatAnthropic
from langgraph.graph import END, StateGraph

from fluffy_potato_curriculum.common.config import get_settings, require_anthropic_key

# The course anchor. L03 keeps the model CONSTANT -- Sonnet only -- so the *node* is the only
# new variable. (Per-node model mixing is L04's mechanism, not L03's.)
SONNET = "claude-sonnet-4-6"

# A few raw support tickets -- the running example. The one node's single job is to EXTRACT a
# handful of structured fields out of one of these short blobs of text. They deliberately vary
# in WHICH fields are present and WHAT the issue is, so the extractor is exercised, not memorised.
TICKETS = {
    # a clean ticket: every field is clearly present
    "clear": (
        "Order #48213 -- I was charged twice for my annual subscription on 2026-06-30. "
        "Please refund the duplicate charge. -- Dana Ruiz, dana.ruiz@example.com"
    ),
    # a harder ticket: no order number, no email -- the node must cope with missing fields
    "ambiguous": (
        "hey the export button keeps throwing a 500 error whenever i click it on the reports "
        "page, been happening all week, can someone look?"
    ),
    # email present, but no order number -- a login/account problem, not billing
    "login": (
        "Can't log into my account since the weekend -- the password-reset emails never "
        "arrive. Anything you can do? Reach me at priya.natarajan@example.com."
    ),
    # order number present, but no email -- and it's a feature request, not a problem
    "feature_request": (
        "Order #77120 here. Love the app! Any chance you could add a dark mode to the mobile "
        "version? Would make late-night use so much easier. -- Marcus"
    ),
    # all fields present again, but a different issue type (shipping / physical damage)
    "damaged": (
        "Order #55901 arrived damaged -- the box was crushed and the screen is cracked. "
        "Please advise on a replacement. Reachable at sam.okafor@example.com."
    ),
}

# The fields the extract node pulls out. Asking for a fixed SHAPE (not free text) is the L02
# structured-output-by-instruction discipline, now returned as a typed state update.
EXTRACT_FIELDS = ["order_id", "issue_type", "customer_email"]

# Live calls load the key through the config seam (common/config.py) -- same as L01-L02, only
# the *client* is the framework's now. Build the client only when a key is present so a keyless
# restart-and-run-all still passes; the invoke/stream cells below are gated on LIVE.
LIVE = bool(get_settings().anthropic_api_key)
if LIVE:
    # ONE client, used by the ONE node. Kept small (low max_tokens) so a class run costs cents.
    sonnet = ChatAnthropic(model=SONNET, api_key=require_anthropic_key(), max_tokens=256)
else:
    print("No ANTHROPIC_API_KEY set -- build/diagram cells run; invoke/stream cells will skip.")

[↑ Back to top](#top)

## 2. The "before": a plain L01/L02-style call

Here is everything you already know how to do: a bare function that takes a string, builds a
structured-output prompt, calls the model, and returns a parsed `dict`. This is the **"before"**
-- keep it on screen. In section 4 we turn *this* into a node and walk the diff.

The `parse_fields` helper is the **defensive parsing** discipline from L02: try `json.loads`
first, fall back to a regex that finds the first `{...}` block. We reuse it in the node too.

In [ ]:
def parse_fields(text: str) -> dict[str, object]:
    """Parse the model's reply into a dict, defensively (the L02 discipline).

    Try strict JSON first; if the model wrapped the JSON in prose, fall back to the first
    ``{...}`` block. If both fail, return an empty dict rather than raising -- a node should
    return a well-typed *shape* even when the model misbehaves.

    Example input:
        '{"order_id": "48213", "issue_type": "billing", "customer_email": "dana@x.com"}'
    Example output:
        {"order_id": "48213", "issue_type": "billing", "customer_email": "dana@x.com"}
    """
    try:
        return dict(json.loads(text))
    except (json.JSONDecodeError, TypeError, ValueError):
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            try:
                return dict(json.loads(match.group(0)))
            except (json.JSONDecodeError, TypeError, ValueError):
                return {}
        return {}


def extract_prompt(raw_text: str) -> str:
    """Build the structured-output prompt: ask for a fixed JSON shape (L02 discipline)."""
    fields = ", ".join(EXTRACT_FIELDS)
    return (
        "Extract these fields from the support ticket and reply with ONLY a JSON object "
        f"with exactly these keys: {fields}. "
        "Use null for any field not present in the ticket.\n\n"
        f"Ticket:\n{raw_text}"
    )


def extract_plain(raw_text: str) -> dict[str, object]:
    """The L01/L02 shape: string in, parsed dict out. No graph, no state -- the "before"."""
    reply = sonnet.invoke(extract_prompt(raw_text))
    return parse_fields(str(reply.content))


if LIVE:
    print(extract_plain(TICKETS["clear"]))
else:
    print(
        "(skipped -- no key) extract_plain would return e.g. "
        "{'order_id': '48213', 'issue_type': 'billing', 'customer_email': 'dana.ruiz@example.com'}"
    )

[↑ Back to top](#top)

## 3. The typed state

The **state** is a typed object passed into the node and updated by its return value. Keep it
small: one **input** field the node reads (`raw_text`) and one **output** field the node
populates (`extracted_fields`). That's the smallest state that makes the point.

Why a `TypedDict` and not a bare `dict`? The typed shape is the **contract** a *future* node
(L04) can read without opening this node's source -- it says exactly what a node may read and
must return. A plain dict loses that; it's discoverable only by inspection, not by type. For
*one* node the payoff isn't visible yet -- that's honest, and it's the point of L04.

In [ ]:
class ExtractState(TypedDict):
    """State threaded through the one-node extract graph.

    Example value after a run:
        {"raw_text": "Order #48213 -- I was charged twice...",
         "extracted_fields": {"order_id": "48213", "issue_type": "billing",
                              "customer_email": "dana.ruiz@example.com"}}
    """

    raw_text: str  # the raw incoming ticket (the INPUT the node reads)
    extracted_fields: dict[str, object]  # the OUTPUT the node populates

[↑ Back to top](#top)

## 4. The node: one LLM call, wrapped

Now the same underlying call becomes a **node**: a plain typed function `state -> update`. Two
things change from the `extract_plain` "before", and nothing else does:

1. **The signature.** It takes the *state* (not a bare string) and returns a **state update** --
   a dict of *only the fields it changed* (`{"extracted_fields": ...}`), **not** the whole state
   and **not** "the answer". LangGraph merges that update back into the shared state.
2. **Nothing else.** Same prompt (`extract_prompt`), same defensive parser (`parse_fields`),
   same `ChatAnthropic` client. The *work* is identical; only the *wrapping* is new.

This is the **purity contract**: given the same `raw_text`, the node does the same unit of work
and returns the same *shape* of update -- no hidden reads, no side effects beyond the one LLM
call. (The model's sampling is non-deterministic; the contract is about what the function reads
and returns, not the tokens the model picks.)

In [ ]:
def extract_node(state: ExtractState) -> dict[str, object]:
    """Extract structured fields from the ticket. ONE unit of work.

    state in  -> reads state["raw_text"]
    update out -> returns ONLY {"extracted_fields": ...}, the field it changed.
                  It does NOT return the whole state, and NOT "the answer" -- an *update*.
    """
    reply = sonnet.invoke(extract_prompt(state["raw_text"]))
    parsed = parse_fields(str(reply.content))
    return {"extracted_fields": parsed}

[↑ Back to top](#top)

## 5. Compile the smallest possible graph

The `StateGraph` builder at its smallest: declare the state schema, `add_node` for the one
node, `set_entry_point`, wire the node straight to `END`, and `compile()`. There is exactly
**one node** and **one edge**, and that edge goes straight to `END` -- there is nothing to
wire *between* nodes yet, on purpose.

`compile()` turns the *declaration* into a *runnable* -- nothing has executed yet. The diagram
render needs no API key: the control flow is already data you can inspect (a one-node diagram
is nearly content-free -- L04 is where the diagram earns its keep).

In [ ]:
builder = StateGraph(ExtractState)
builder.add_node("extract", extract_node)
builder.set_entry_point("extract")
builder.add_edge("extract", END)  # the node's only edge: straight to END
extract_app = builder.compile()

# The compiled graph can draw itself. One node, one edge to END -- that's the whole graph.
print(extract_app.get_graph().draw_mermaid())

[↑ Back to top](#top)

## 6. Invoke and inspect one node

`invoke()` runs the compiled graph on a specific input. Watch the returned state: the
`raw_text` field is **still there unchanged**, and `extracted_fields` is now **populated**. That
is "state comes out" made concrete -- `invoke()` returns the *whole state* (input intact, output
added), not just the node's output.

Run a representative few -- `clear` (all fields present), `feature_request` (an order number but
no email), and `ambiguous` (neither) -- to span **full -> partial -> mostly-missing** fields.
`extracted_fields` comes back as a well-typed dict every time; the defensive parser and the
`null`-for-missing prompt keep the *shape* stable while the extracted *values* vary with what
each ticket actually contains. (All five tickets are defined above -- swap any key in to see the
rest.)

In [ ]:
if not LIVE:
    print("No ANTHROPIC_API_KEY set -- skipping the live invoke. Set it to run the node.")
else:
    # All five tickets are defined above; we invoke a representative few to keep a class run
    # cheap (each is one live call). Swap in any other key -- e.g. "login", "damaged" -- to see
    # the rest. The trio below spans full -> partial -> mostly-missing fields.
    for name in ("clear", "feature_request", "ambiguous"):
        result = extract_app.invoke({"raw_text": TICKETS[name]})
        print(f"=== {name} ticket ===")
        print("raw_text still present:", result["raw_text"][:48], "...")
        print("extracted_fields     :", result["extracted_fields"])
        print()

[↑ Back to top](#top)

## 7. Watch it run with `stream()`

LangGraph's own **built-in** way to watch a run: `stream()` yields one chunk per node as it
fires. For *one* node it's almost overkill -- you'll see a single step -- but it is the same
mechanism L04 uses to watch several nodes fire in sequence.

**This is not tracing.** `stream()` shows *that* the node ran and what it returned; it is not the
structured, comparable trace that real trace tooling gives you. Reading a full run as a trace,
comparing runs, diagnosing failures from a trace alone -- that's **L12's** job. For one node,
`invoke()`'s return value and this stream are all you need to answer *"did my node run, and what
came back?"*

In [ ]:
if not LIVE:
    print("No ANTHROPIC_API_KEY set -- skipping stream(). Set it to watch the single step fire.")
else:
    # One node -> one chunk. Each chunk maps node-name -> that node's state update.
    for chunk in extract_app.stream({"raw_text": TICKETS["clear"]}):
        print(chunk)

[↑ Back to top](#top)

## 8. Takeaways

- **A node is one LLM call you wire.** The state schema, the `StateGraph` ceremony, and
  compile/invoke all exist to make that one call *reusable and inspectable* rather than a one-off
  function.
- **State goes in, state comes out.** The node returns an *update* to shared state
  (`{"extracted_fields": ...}`), not "the answer". `invoke()` hands back the whole state -- input
  intact, output added.
- **The node is a pure function over state.** Same `raw_text` in, same *shape* of update out; no
  hidden reads, no side effects beyond the one LLM call.
- **First framework, native client.** The call goes through LangChain's `ChatAnthropic`, not the
  `potato_llm` seam. Both are still "just an API call" underneath.
- **More ceremony than a plain function -- and the payoff isn't here yet.** For one node,
  `extract_app.invoke(...)` is more setup than `extract_plain(...)`. It pays off in **L04**, when
  the *same* ceremony scales to many nodes with no per-node redesign.

> **You just wired one step -- next lesson, you wire several of them together.**

[↑ Back to top](#top)